# QEvict on DeepSeek-V2-Lite-Chat: LongBench, 10% KV-cache budget (Colab)

Runs **QEvict at a 10% KV-cache budget** on **`deepseek-ai/DeepSeek-V2-Lite-Chat`** over the paper's Table 2 LongBench
tasks except NarrativeQA: Qasper, MultiFieldQA-en, HotpotQA, 2WikiMQA, MuSiQue, GovReport, QMSum, MultiNews, TREC,
TriviaQA, SAMSum (every example).

**Settings, as in the paper** (Table 2; Appendix B.5 and C): `cache_budget 0.10`, window W = 8, routing every R = 8 steps,
5 sink tokens, 128 recent tokens, alpha = 0.70 of the historical byte budget to INT2 (`bytes` mode), read ratio
rho = 0.25, bidirectional F<->Q migration, greedy decoding, THUDM prompts, metrics and post-processing.
Prompts are middle-truncated to 31,500 tokens (DeepSeek-V2-Lite is a 32k model; THUDM's convention for 32k models).

**Batch size is 1**, one example per `generate()` call, as in the paper's LongBench runs (Appendix H.6). Batches of
16, 10, 8 or 4 cannot be run without changing the method: the cache batches only equal-length prompts and the MLA
forward rejects padding, while every LongBench prompt has a different length. Memory rules it out too: at 31.5k tokens
the prefill's fp32 logits are ~13 GB per row, so 4 rows alone need ~52 GB on top of ~31 GB of weights.

**Needs an 80 GB GPU** (A100 80GB). At a 31.5k-token prompt the model computes fp32 logits for every prompt
position (~13 GB) on top of ~31 GB of bf16 weights.

Results go to Google Drive, one task at a time; rerun all cells to resume (finished tasks are skipped).

In [ ]:
# ===================== Settings =====================
BRANCH         = "Clustered_int2_paused_fast_deepseek"
REPO_URL       = "https://github.com/shaswata965-coder/RecoverKv.git"
PIN_COMMIT     = "9bda7d291e8f50dc4c145351c6410d01b96a7742"
MODEL_ID       = "deepseek-ai/DeepSeek-V2-Lite-Chat"
MODEL_REVISION = "85864749cd611b4353ce1decdb286193298f64c7"
MODEL_DIR      = "/content/models/DeepSeek-V2-Lite-Chat"   # keep "Chat" in the name: it turns the chat template on
DRIVE_ROOT     = "/content/drive/MyDrive/qevict_deepseek_v2_lite_longbench_b010"
RUN_FULLKV     = False   # True also runs the Full-KV (100%) reference row, at about the same cost again

TASKS = ["qasper", "multifieldqa_en", "hotpotqa", "2wikimqa", "musique",
         "gov_report", "qmsum", "multi_news", "trec", "triviaqa", "samsum"]
LABEL = {"qasper": "Qasper", "multifieldqa_en": "Mul.QA", "hotpotqa": "Hot.QA", "2wikimqa": "2Wi.QA",
         "musique": "Musique", "gov_report": "Gov.Re.", "qmsum": "QMSum", "multi_news": "M.News",
         "trec": "TREC", "triviaqa": "Tri.QA", "samsum": "SAMSum"}
QEVICT = {"cache_budget": 0.10, "window_size": 8, "num_sink_tokens": 5, "local_window_size": 128,
          "quant_ratio": 0.70, "quant_budget_mode": "bytes", "quant_gate_ratio": 0.25,
          "first_eviction_step": 0}
MAX_LENGTH = 31500
ARMS = {"qevict_b010": ("configs/longbench_deepseek_v2_lite_flash.yaml",
                        [f"cache.{k}={v}" for k, v in QEVICT.items()]),
        "fullkv":      ("configs/longbench_deepseek_v2_lite_full_cache.yaml", [])}
RUN_ARMS = ["qevict_b010"] + (["fullkv"] if RUN_FULLKV else [])

In [ ]:
import subprocess, json, os, sys, shutil, time, zipfile, csv, statistics
from pathlib import Path

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
                     capture_output=True, text=True).stdout.strip().splitlines()[0].split(",")
print("GPU:", gpu[0].strip(), f"{float(gpu[1]) / 1024:.0f} GB")
if float(gpu[1]) / 1024 < 75:
    raise RuntimeError("This needs an 80 GB GPU: a 31.5k-token prompt does not fit in 40 GB.")

from google.colab import drive
drive.mount("/content/drive")
ROOT = Path(DRIVE_ROOT)
ROOT.mkdir(parents=True, exist_ok=True)

REPO = Path("/content/RecoverKv")
if not REPO.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(["git", "fetch", "origin", BRANCH], cwd=REPO, check=True)
subprocess.run(["git", "checkout", "--force", PIN_COMMIT], cwd=REPO, check=True)
print("commit", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip())

## Install the pinned stack

A Python 3.12 venv with the stack the recorded GPU runs used: torch 2.6.0 (+cu124; the eviction's `torch.compile`
fails on 2.7.1), transformers 4.47.1 (the cache refuses newer), and a prebuilt flash-attn wheel that loads against
torch 2.6.0 (2.7.4.post1; the 2.8.x wheels for torch 2.6 fail on a C++ ABI symbol).

In [ ]:
VENV = Path("/content/py312_env")
PY = str(VENV / "bin" / "python")
PINS = {"torch": "2.6.0", "torchvision": "0.21.0", "torchaudio": "2.6.0", "triton": "3.2.0",
        "transformers": "4.47.1"}
STACK_ID = json.dumps({"python": "3.12", "cuda": "cu124", **PINS}, sort_keys=True)
MARK = VENV / ".recoverkv_stack"
_DROP = ("PYTHONPATH", "PYTHONHOME", "PYTHONSTARTUP", "CPATH", "C_INCLUDE_PATH", "CPLUS_INCLUDE_PATH",
         "PIP_CONSTRAINT", "MPLBACKEND")
VENV_ENV = {k: v for k, v in os.environ.items() if k not in _DROP}
VENV_ENV.update(VIRTUAL_ENV=str(VENV), PATH=f"{VENV / 'bin'}:{os.environ.get('PATH', '')}", MPLBACKEND="Agg")

def sh(*cmd, env=None):
    subprocess.run([str(c) for c in cmd], check=True, env=VENV_ENV if env is None else env)

def vpip(*a):
    sh(PY, "-m", "pip", "install", "-q", "--disable-pip-version-check", *a)

def versions():
    code = ("import json, importlib.metadata as m\nout = {}\n"
            "for d in ('torch','torchvision','torchaudio','triton','transformers','flash_attn'):\n"
            "    try: out[d] = m.version(d)\n    except m.PackageNotFoundError: out[d] = None\n"
            "print(json.dumps(out))")
    v = json.loads(subprocess.check_output([PY, "-c", code], env=VENV_ENV, text=True))
    bad = {k: v.get(k) for k, want in PINS.items() if (v.get(k) or "").split("+")[0] != want}
    if bad:
        raise RuntimeError(f"the venv drifted from the pinned stack: {bad}. Delete {VENV} and rerun.")
    return v

def fa_loads():
    return subprocess.run([PY, "-c", "import torch, flash_attn_2_cuda, flash_attn"], env=VENV_ENV,
                          capture_output=True).returncode == 0

if not (MARK.exists() and MARK.read_text() == STACK_ID and Path(PY).exists()):
    shutil.rmtree(VENV, ignore_errors=True)
    if not (shutil.which("python3.12") and subprocess.run(["python3.12", "-c", "import ensurepip, venv"],
                                                          capture_output=True).returncode == 0):
        apt_env = dict(os.environ, DEBIAN_FRONTEND="noninteractive")
        sh("apt-get", "update", "-qq", env=apt_env)
        sh("apt-get", "install", "-y", "-qq", "python3.12", "python3.12-venv", "python3.12-dev", env=apt_env)
    sh("python3.12", "-m", "venv", VENV, env=os.environ)
    constraints = VENV / "constraints.txt"
    constraints.write_text("".join(f"{k}=={v}\n" for k, v in PINS.items()))
    vpip("--upgrade", "pip")
    vpip("torch==2.6.0", "torchvision==0.21.0", "torchaudio==2.6.0",
         "--index-url", "https://download.pytorch.org/whl/cu124")
    vpip("-c", constraints,
         "transformers==4.47.1", "accelerate>=0.25,<2", "datasets>=2.16,<3", "huggingface_hub>=0.24,<1.0",
         "sentencepiece", "protobuf", "einops>=0.7", "scipy", "pyyaml", "tqdm",
         "rouge", "fuzzywuzzy", "python-Levenshtein", "jieba")
    versions()
    tv, abi, py = subprocess.check_output([PY, "-c",
        "import torch, sys; print(torch.__version__.split('+')[0]); "
        "print(int(torch._C._GLIBCXX_USE_CXX11_ABI)); "
        "print(f'cp{sys.version_info.major}{sys.version_info.minor}')"], env=VENV_ENV, text=True).split()
    tmm = ".".join(tv.split(".")[:2])
    for v in ("2.7.4.post1", "2.8.3", "2.8.2"):
        u = (f"https://github.com/Dao-AILab/flash-attention/releases/download/v{v}/"
             f"flash_attn-{v}+cu12torch{tmm}cxx11abi{'TRUE' if abi == '1' else 'FALSE'}-{py}-{py}-linux_x86_64.whl")
        try:
            vpip("--no-deps", "-c", constraints, u)
        except subprocess.CalledProcessError:
            continue
        if fa_loads():
            print("flash-attn wheel:", u.rsplit("/", 1)[1])
            break
        sh(PY, "-m", "pip", "uninstall", "-y", "-q", "flash-attn")
    else:
        raise RuntimeError(f"no prebuilt flash-attn wheel loads against torch {tv} / {py} / cxx11abi={abi}")
    MARK.write_text(STACK_ID)
print("venv:", versions())

## Model and data

In [ ]:
from huggingface_hub import snapshot_download, hf_hub_download
snapshot_download(MODEL_ID, revision=MODEL_REVISION, local_dir=MODEL_DIR,
                  allow_patterns=["*.json", "*.safetensors", "*.py", "tokenizer*"])
# DeepSeek-V2-Lite-Chat at the pinned revision: same remote code the repo's MLA forward was checked against,
# and a chat template (the runner wraps 8 of the 11 tasks in it, as THUDM does).
import hashlib
for f in ("config.json", "configuration_deepseek.py", "modeling_deepseek.py"):
    h = lambda p: hashlib.sha256(p.read_bytes()).hexdigest()
    assert h(Path(MODEL_DIR) / f) == h(REPO / "models/deepseek_v2/upstream" / f), f"{f}: not DeepSeek-V2-Lite"
assert json.loads((Path(MODEL_DIR) / "tokenizer_config.json").read_text()).get("chat_template"), "no chat template"
assert "chat" in Path(MODEL_DIR).name.lower(), "MODEL_DIR's name must contain 'chat' (it switches the template on)"
print("model:", MODEL_ID, MODEL_REVISION[:12])

LB_DIR = Path("/content/longbench")
LB_DIR.mkdir(parents=True, exist_ok=True)
if not all((LB_DIR / f"{t}.jsonl").exists() for t in TASKS):
    with zipfile.ZipFile(hf_hub_download("THUDM/LongBench", "data.zip", repo_type="dataset")) as z:
        for t in TASKS:
            member = next(n for n in z.namelist() if n.endswith(f"/{t}.jsonl") or n == f"{t}.jsonl")
            with z.open(member) as src, open(LB_DIR / f"{t}.jsonl", "wb") as dst:
                shutil.copyfileobj(src, dst)
N = {t: sum(1 for l in open(LB_DIR / f"{t}.jsonl") if l.strip()) for t in TASKS}
print(N)

ENV = dict(VENV_ENV, PYTHONHASHSEED="0", CUBLAS_WORKSPACE_CONFIG=":4096:8", TOKENIZERS_PARALLELISM="false",
           HF_HUB_OFFLINE="1", LONGBENCH_LOCAL_DIR=str(LB_DIR), PYTHONUNBUFFERED="1")

## Run

One process per arm runs every task that is not finished yet. A task is finished when its `<task>.meta.json` is on
Drive (the runner writes it last). Each QEvict task's sidecar must report `read_gate: gated`.

In [ ]:
def run_arm(arm):
    config, extra = ARMS[arm]
    out = ROOT / arm
    out.mkdir(parents=True, exist_ok=True)
    todo = [t for t in TASKS if not (out / f"{t}.meta.json").exists()]
    if todo:
        cmd = [PY, "main.py", "--config", config, "--override",
               f"model.name={MODEL_DIR}", f"model.revision={MODEL_REVISION}",
               f"longbench.datasets=[{','.join(todo)}]", f"longbench.output_dir={out}",
               "longbench.num_samples=max", f"longbench.max_length={MAX_LENGTH}",
               "longbench.resume=false", "longbench.skip_oom=false", "run.seed=42", *extra]
        print("$", " ".join(cmd))
        with open(ROOT / f"{arm}.log", "a", buffering=1) as log:
            p = subprocess.Popen(cmd, cwd=REPO, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                 text=True, bufsize=1)
            for line in p.stdout:
                log.write(line)
                if any(k in line for k in ("examples,", "examples done", "read gate", "Error", "Traceback")):
                    print("   ", line.rstrip()[:200])
            if p.wait() != 0:
                raise RuntimeError(f"{arm} stopped; see {ROOT / (arm + '.log')}. Rerun this cell to resume.")
    for t in TASKS:
        meta = json.loads((out / f"{t}.meta.json").read_text())
        verdict = meta["read_gate"]["verdict"]
        want = "gated" if arm == "qevict_b010" else "not-expected"
        assert verdict == want and meta["num_examples"] == N[t], (arm, t, verdict, meta["num_examples"])
        print(f"{arm:12s} {t:16s} {meta['num_examples']:4d} examples  read_gate={verdict}")

for arm in RUN_ARMS:
    run_arm(arm)

## Scores

In [ ]:
rows = []
for arm in RUN_ARMS:
    out_csv = ROOT / f"scores_{arm}.csv"
    subprocess.run([PY, "-m", "modules.evaluation.longbench_scoring", "--predictions_dir", str(ROOT / arm),
                    "--out_csv", str(out_csv)], cwd=REPO, env=ENV, check=True, capture_output=True)
    s = {r["dataset"]: float(r["score"]) for r in csv.DictReader(open(out_csv))}
    v = [s[t] for t in TASKS]
    rows.append(["QEvict" if arm == "qevict_b010" else "Full-KV", "10%" if arm == "qevict_b010" else "100%",
                 *[f"{x:.2f}" for x in v], f"{statistics.mean(v):.2f}"])
head = ["Method", "KV memory", *[LABEL[t] for t in TASKS], "Avg (11)"]
table = ["| " + " | ".join(head) + " |", "|" + "---|" * len(head)] + ["| " + " | ".join(r) + " |" for r in rows]
(ROOT / "table_b010.md").write_text("\n".join(table) + "\n")
print("\n".join(table))
print("\nLaTeX:\n" + "\n".join(" & ".join(r).replace("%", r"\%") + r" \\" for r in rows))